In [ ]:
pip install pyfinviz

In [ ]:
from pyfinviz.news import News

news = News()

# available variables:
print(news.main_url)  # scraped URL
print(news.soup)  # beautiful soup object
print(news.news_df)  # NEWS table information in a pd.DataFrame object
print(news.blogs_df)  # BLOGS table information in a pd.DataFrame object

In [ ]:
from pyfinviz.crypto import Crypto

# with no params (SECTOR, OVERVIEW by default)
crypto = Crypto()
# with params
crypto = Crypto(relative_performance_option=Crypto.RelativePerformanceOption.ONE_YEAR)

# available variables:
print(crypto.main_url)  # scraped URL
print(crypto.soup)  # beautiful soup object
print(crypto.table_df)  # table information in a pd.DataFrame object

In [ ]:
from pyfinviz.groups import Groups

# with no params (sector overview)
groups = Groups()
# with params (View the group VALUATION of the INDUSTRY sector)
groups = Groups(group_option=Groups.GroupOption.INDUSTRY, view_option=Groups.ViewOption.VALUATION)
# with params (View the group PERFORMANCE of the TECH sector)
groups = Groups(group_option=Groups.GroupOption.INDUSTRY_TECHNOLOGY,
                view_option=Groups.ViewOption.PERFORMANCE)

# available variables:
print(groups.main_url)  # scraped URL
print(groups.soup)  # beautiful soup object
print(groups.table_df)  # table information in a pd.DataFrame object

In [ ]:
from pyfinviz.insider import Insider

# with no params (ALL the LATEST insider trades)
insider = Insider()
# with params (the LATEST BUY insider trades)
insider = Insider(filter_option=Insider.FilterOption.BUY)

# available variables:
print(insider.main_url)  # scraped URL
print(insider.soup)  # beautiful soup object
print(insider.table_df)  # table information in a pd.DataFrame object

In [ ]:
from pyfinviz.quote import Quote

quote = Quote(ticker="AMZN")

# available variables:
print(quote.exists)  # check if fetch was successful (STOCK may not exist)
print(quote.ticker)  # AMZN
print(quote.exchange)  # NASD
print(quote.company_name)  # Amazon.com, Inc.
print(quote.sectors)  # ['Consumer Cyclical', 'Internet Retail', 'USA']
print(quote.fundamental_df)  # Index    P/E EPS (ttm) Insider Own  ...  SMA50  SMA200     Volume  Change
print(quote.outer_ratings_df)  # 0   Nov-04-20     Upgrade  ...                Hold → Buy  $3360 → $4000
print(quote.outer_news_df)  # 0   Jan-04-21 10:20PM  ...                   Bloomberg
print(quote.income_statement_df)  # 1      12/31/2019  ...                    22.99206
print(quote.insider_trading_df)  # 0         WILKE JEFFREY A  ...  http://www.sec.gov/Archives/edgar/data/1018724...
print(quote.reuters_income_statement_df) #Available only for elite

In [ ]:
from pyfinviz.screener import Screener

# with no params (default screener table)
#screener = Screener()
# with params (The first 3 pages of "STOCKS ONLY" where Analyst recommend a strong buy)
# print(dir(Screener.BLABLA)) to get info on which attributes does a class have
options = [Screener.IndustryOption.STOCKS_ONLY_EX_FUNDS, Screener.MarketCapOption.LARGE_OVER_USD10BLN]

screener = Screener(filter_options=options, view_option=Screener.ViewOption.VALUATION,
                    pages=[x for x in range(1, 44)])
screenerOV = Screener(filter_options=options, view_option=Screener.ViewOption.OVERVIEW,
                      pages=[x for x in range(1,44)])
screenerF = Screener(filter_options=options, view_option=Screener.ViewOption.FINANCIAL,
                      pages=[x for x in range(1,44)])
screenerP = Screener(filter_options=options, view_option=Screener.ViewOption.PERFORMANCE,
                      pages=[x for x in range(1,44)])
screenerT = Screener(filter_options=options, view_option=Screener.ViewOption.TECHNICAL,
                      pages=[x for x in range(1,44)])

# available variables:
#print(screener.main_url)  # scraped URL
#print(screener.soups)  # beautiful soup object per page {1: soup, 2: soup, ...}
print(screener.data_frames)  # table information in a pd.DataFrame object per page {1: table_df, 2, table_df, ...}

In [ ]:
import pandas as pd
combined_dataOV = pd.concat(screenerOV.data_frames.values(), ignore_index=True)
combined_data = pd.concat(screener.data_frames.values(), ignore_index=True)
combined_dataF = pd.concat(screenerF.data_frames.values(), ignore_index=True)
combined_dataP = pd.concat(screenerP.data_frames.values(), ignore_index=True)
combined_dataT = pd.concat(screenerT.data_frames.values(), ignore_index=True)
print(f"The combined dataset has {combined_dataOV.shape[0]} rows and {combined_dataOV.shape[1]} columns.")
print(f"The combined dataset has {combined_data.shape[0]} rows and {combined_data.shape[1]} columns.")
print(f"The combined dataset has {combined_dataF.shape[0]} rows and {combined_dataF.shape[1]} columns.")
print(f"The combined dataset has {combined_dataP.shape[0]} rows and {combined_dataP.shape[1]} columns.")
print(f"The combined dataset has {combined_dataT.shape[0]} rows and {combined_dataT.shape[1]} columns.")

# Perform successive merges with unique suffixes for each dataset
merged_data = pd.merge(combined_dataOV, combined_data, on="Ticker", how="outer", suffixes=('_OV', '_C'))
merged_data = pd.merge(merged_data, combined_dataF, on="Ticker", how="outer", suffixes=('', '_F'))
merged_data = pd.merge(merged_data, combined_dataP, on="Ticker", how="outer", suffixes=('', '_P'))
merged_data = pd.merge(merged_data, combined_dataT, on="Ticker", how="outer", suffixes=('', '_T'))

# Print the shape of the final merged dataset
print(f"The final merged dataset has {merged_data.shape[0]} rows and {merged_data.shape[1]} columns.")

# Clean dataset
print(merged_data.columns)
# List of columns to drop
columns_to_drop = [
    'No_C', 'MarketCap_C', 'PE_C', 'Price_C', 'Change_C', 'Volume_C',
    'No', 'MarketCap', 'Price', 'Change', 'Volume',
    'No_P', 'Price_P', 'Change_P', 'Volume_P',
    'No_T', 'Price_T', 'Change_T', 'Volume_T'
]

# Drop the specified redundant columns
merged_data = merged_data.drop(columns=columns_to_drop)

# Rename columns by dropping the "_OV" suffix
columns_to_rename = {
    'No_OV': 'No',
    'MarketCap_OV': 'MarketCap',
    'PE_OV': 'PE',
    'Price_OV': 'Price',
    'Change_OV': 'Change',
    'Volume_OV': 'Volume'
}

# Apply the renaming
merged_data.rename(columns=columns_to_rename, inplace=True)

# Print the updated column names to confirm the changes
print("Updated column names:")
print(merged_data.columns)

# Print the shape of the cleaned dataset to confirm the changes
print(f"The cleaned dataset has {merged_data.shape[0]} rows and {merged_data.shape[1]} columns.")

# Save the merged dataset to a CSV file
dataset = "merged_data.csv"  # Specify the desired file name
merged_data.to_csv(dataset, index=False)
print(f"Merged data saved as '{dataset}' in the current working directory.")